# 6.18 · 非负矩阵分解 / Non-negative Matrix Factorization (NMF)

> **课程定位 / Where this fits**
> Part 6 收官。PCA(6.8)分解出的成分有正有负, 难解释("负的词频"是什么?)。NMF 要求分解出的因子**全非负**, 于是把数据表示成一堆**非负部件的叠加(相加而非相消)**——天然可解释: 文档=主题之和、人脸=部件之和。是主题建模、推荐(评分矩阵分解)、信号分离的利器。
> NMF factorizes non-negative data into non-negative parts that add up (no cancellation) — yielding interpretable "parts": topics in text, parts in faces. Closes Part 6.

> 💡 **面试相关 / Interview-relevant**
> - "NMF 与 PCA 区别 / NMF 为何更可解释" ★★★★★（非负=部件叠加, 无相消）
> - "NMF 的目标与约束(min ‖X-WH‖, W,H≥0)" ★★★★
> - "NMF 用在哪(主题模型/推荐/源分离)" ★★★★
> - "NMF 的非唯一性 / 初始化敏感" ★★★

---

## 学习目标 / Learning Objectives
1. NMF 分解 $\mathbf{X}\approx\mathbf{W}\mathbf{H}$, $\mathbf{W},\mathbf{H}\ge0$。
2. 非负约束为何带来可解释的"部件"。
3. 文本主题建模(20 Newsgroups)。
4. NMF vs PCA 的成分对比。

## 目录 / TOC
1. [分解与非负约束 ⭐](#1)
2. [📰 数据: 20 Newsgroups + 主题 ⭐](#2)
3. [NMF vs PCA 成分 ⭐](#3)
4. [选成分数 + 应用](#4)
5. [Part 6 全景回顾](#5)


<a id="1"></a>
## 1. 分解与非负约束 ⭐ / Factorization & Non-negativity

NMF 把一个**非负**矩阵 $\mathbf{X}$($n\times d$, 如 n 文档 × d 词)近似分解成两个非负矩阵之积:
$$\mathbf{X} \approx \mathbf{W}\mathbf{H}, \qquad \mathbf{W}\ge0\ (n\times k),\ \mathbf{H}\ge0\ (k\times d)$$
- $\mathbf{H}$ 的每一行 = 一个**部件/主题**(在原特征上的非负权重, 如"这个主题里哪些词重要")。
- $\mathbf{W}$ 的每一行 = 该样本由各部件**以非负强度叠加**的系数(如"这篇文档由各主题混合的比例")。

目标: 最小化重构误差 $\|\mathbf{X}-\mathbf{W}\mathbf{H}\|_F^2$ s.t. $\mathbf{W},\mathbf{H}\ge0$, 用乘性更新或坐标下降迭代。

**非负约束为何关键(vs PCA)**: PCA 成分可正可负, 重构靠**正负相消**, 单个成分难解释。NMF 只能**相加**——每个部件只能往上叠, 于是学到的是真实"组成部件"(主题/五官), **天然可解释**。代价: 解**不唯一**、对初始化敏感。


<a id="2"></a>
## 2. 数据: 20 Newsgroups + 主题 ⭐ / Topic Modeling

复用 **20 Newsgroups**(5.13 介绍过)。取几个主题的帖子, TF-IDF 向量化(全非负), NMF 分解成 k 个主题, 看每个主题的关键词——这就是经典的**主题模型**。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
sns.set_theme(style="whitegrid")

cats = ['rec.sport.baseball', 'sci.space', 'comp.graphics', 'talk.politics.mideast', 'sci.med']
news = fetch_20newsgroups(subset='train', categories=cats,
                          remove=('headers','footers','quotes'), random_state=0)
print(f"20 Newsgroups: {len(news.data)} 篇, {len(cats)} 个真实主题")

vec = TfidfVectorizer(max_features=2000, stop_words='english', min_df=5, max_df=0.5)
X = vec.fit_transform(news.data)        # 文档×词 TF-IDF (非负)
words = np.array(vec.get_feature_names_out())
print(f"TF-IDF 矩阵: {X.shape} (全非负 → 适合 NMF)")

k = 5
nmf = NMF(n_components=k, init="nndsvda", random_state=0, max_iter=400).fit(X)
print(f"\nNMF 分解出 {k} 个主题, 各主题 Top 词:")
for i, comp in enumerate(nmf.components_):
    top = words[comp.argsort()[-8:][::-1]]
    print(f"  主题 {i}: {', '.join(top)}")
print("\n无监督地, NMF 把词聚成可解释的主题(体育/太空/图形/中东/医学)")


<a id="3"></a>
## 3. NMF vs PCA 成分 ⭐ / Components vs PCA

把同样的数据用 PCA(=TruncatedSVD, 文本上用它)分解, 对比成分的可解释性: NMF 成分全非负(纯"加成"的主题词), PCA 成分有正有负(混着"反相关"词, 难当主题读)。


In [ ]:
from sklearn.decomposition import TruncatedSVD
svd = TruncatedSVD(n_components=k, random_state=0).fit(X)

print("PCA/SVD 成分: 含正负权重, 单个成分难解释为'主题'")
for i, comp in enumerate(svd.components_[:3]):
    top_pos = words[comp.argsort()[-5:][::-1]]
    top_neg = words[comp.argsort()[:5]]
    print(f"  成分{i}: +[{', '.join(top_pos)}]  -[{', '.join(top_neg)}]")
print("\nNMF 成分(对比): 全非负, 每个就是一组共同出现的词 = 干净的主题")
for i, comp in enumerate(nmf.components_[:3]):
    print(f"  主题{i}: {', '.join(words[comp.argsort()[-5:][::-1]])}")
print("\n→ 非负约束让 NMF 成分=可解释部件; PCA 的正负混合成分难直接读成主题")


<a id="4"></a>
## 4. 选成分数 + 应用 / Choosing k & Applications

- **选 k**: 看重构误差(`reconstruction_err_`)随 k 的变化找肘部, 或按业务想要的主题数定。
- **应用**: 主题建模(本例)、**推荐系统**(用户×物品评分矩阵分解成用户/物品隐因子, 都非负)、图像部件分解(人脸→五官)、音频源分离、文档聚类(W 当降维特征)。


In [ ]:
ks = range(2, 11)
errs = [NMF(n_components=k, init="nndsvda", random_state=0, max_iter=300).fit(X).reconstruction_err_ for k in ks]
fig, ax = plt.subplots(figsize=(7,4))
ax.plot(list(ks), errs, "o-")
ax.set_xlabel("成分数 k (主题数)"); ax.set_ylabel("重构误差 ‖X-WH‖")
ax.set_title("NMF 重构误差随 k 下降; 取肘部或按需要的主题数定 k")
plt.tight_layout(); plt.show()

# 文档的主题分布 = W 的一行 / a document's topic mixture
W = nmf.transform(X[:1])
print(f"第1篇文档(真实类别: {news.target_names[news.target[0]]})的主题强度分布:")
print("  ", (W[0]/W[0].sum()).round(2), "→ 主要属主题", W[0].argmax())
print("W 给出每篇文档的主题混合比例; 可当降维特征或软聚类")


<a id="5"></a>
## 5. Part 6 全景回顾 / Part 6 Big Picture

```
无监督学习 = 没有标签, 发现数据自身结构:

聚类(发现分组):
  K-Means(6.1)/Mini-batch(6.2): 球形, 快, 硬分配
  层次(6.3): 树状图, 不预设K
  DBSCAN(6.4)/HDBSCAN(6.5): 密度, 任意形状+噪声, HDBSCAN 适应不同密度
  GMM(6.6): 软分配+椭圆, EM 算法
  谱聚类(6.7): 图拉普拉斯, 非凸簇
降维(发现紧凑表示):
  线性: PCA(6.8, 方差) / LDA(6.14, 可分性, 有监督) / FA(6.10, 概率潜因子) / NMF(6.18, 非负部件)
  非线性: 核PCA(6.9) / t-SNE(6.12, 可视化) / UMAP(6.13, 快+可transform) / 自编码器(6.15, 深度)
  ICA(6.11): 独立成分, 盲源分离
异常检测(6.16): IsolationForest/OneClassSVM/LOF/重构误差
关联规则(6.17): 购物篮, Apriori/FP-Growth, support/confidence/lift

评估主线: 无标签时用 silhouette(聚类)/解释方差(PCA)/重构误差/BIC(GMM)
缩放主线: 距离类方法(K-Means/DBSCAN/谱/KNN)必须缩放
```

### 💡 面试速查
1. **NMF: X≈WH, W,H≥0** → 部件相加(无相消)→ 可解释主题/部件
2. **vs PCA**: PCA 正交+正负相消(难解释), NMF 非负+加性(可解释); NMF 解不唯一
3. **应用**: 主题模型、推荐(评分矩阵分解)、图像部件、源分离
4. **选 k**: 重构误差肘部 / 业务主题数
5. 适用于**非负数据**(词频/像素/评分); 文本主题建模经典工具

### Part 6 完成 🎉
无监督学习(聚类/降维/异常/关联)全部走通。下一部分 **Part 7 模型评估与优化**——把前面所有模型的评估、调参、特征选择系统化。
